# 03 — Feature extraction experiments

We build a compact feature vector: **log band power** around each flicker frequency (Welch PSD), averaged across channels in `welch_psd`.

This mirrors what `realtime.inference.RealtimeSSVEPDecoder` uses for each analysis window.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np

from acquisition.brainflow_stream import BrainFlowStream
from features.frequency_features import extract_ssvep_feature_vector
from signal_processing.filters import bandpass_filter
from utils.config import SSVEPConfig

In [ ]:
cfg = SSVEPConfig(project_root=ROOT)
fs = BrainFlowStream().sampling_rate()
rng = np.random.default_rng(1)
n = int(cfg.window_seconds * fs)
t = np.arange(n) / fs

left_like = np.sin(2 * np.pi * cfg.left_hz * t)
right_like = np.sin(2 * np.pi * cfg.right_hz * t)

low, high = cfg.bandpass_band_hz()

def feats(x1d):
    eeg = np.stack([x1d, x1d, x1d], axis=0) + 0.1 * rng.standard_normal((3, n))
    eeg = bandpass_filter(eeg, fs, low, high)
    return extract_ssvep_feature_vector(eeg, fs, (cfg.left_hz, cfg.right_hz))

print("Features @ ~left freq:", feats(left_like))
print("Features @ ~right freq:", feats(right_like))

## Next steps

- Try different **window lengths** (`SSVEPConfig.window_seconds`) and compare class separation.
- Log features to disk during calibration for offline hyperparameter search.